# Lesson 10
Run lessons 1–9 first, or use 00_run_all.ipynb. This is the actual teaching source, not a placeholder.

# 第10课 严格验证、完整项目与GitHub交付

冻结配置后做2019—2022逐年扩展窗口：内层仅用过去一年选择Ridge alpha，标签成熟时间清除到外层起点前5个交易日；同日所有资产在一折。候选、失败、否决全部登记，不按外层结果重新改参。

**最终留出真实状态：** 2023—2026历史区间曾在第一课旧版本展示，2021—2022也已反复用于开发评估。当前没有可恢复为未观察的历史最终测试；本课如实报告嵌套时间评估和冻结开发诊断，不能伪造独立样本外证据。未来新增数据只能在冻结后另行观察。

主线把趋势、财务因子、文本、ML、RL同资产目标先合成，再按第六课约束在一个账户只扣一次费用。第2课残差作为ML/RL市场状态；原双腿多空账户另列。去文本/去ML/去RL/去状态过滤逐一消融，并提供过去验证期冻结的风险匹配对照，解释敞口差异。

日收益区块bootstrap保留局部序列依赖，配对比较共享日期。DSR用日Sharpe、偏度、非超额峰度和试验数敏感性；历史试验总数不全、候选相关且样本有限，DSR仅作诊断。逐笔执行和零初始资本期权对冲各用适当指标，不能混算股票Sharpe。

云端项目包包括00_run_all、10课入口、src、固定依赖与配置、来源哈希、离线测试、完整实验台账、英文README与研究报告、静态图表、检查点、1—10课回顾。原始价格、申报全文和受限词典不再分发；本地课程目录不增加文件。

简历只引用已实现并验证的指标。模拟微观结构、历史回测、模拟期权对冲、TPU计算与真实资金交易分开；教学工程不能冒充个人原创研究或未见数据证明。

参考：[DSR原文](https://www.davidhbailey.com/dhbpapers/deflated-sharpe.pdf)、[时间验证](https://scikit-learn.org/stable/modules/cross_validation.html)。


In [ ]:
# 10.01 冻结配置、嵌套扩展窗口与成熟标签隔离
CAPSTONE_CONFIG={'frozen_utc':pd.Timestamp.now(tz='UTC').isoformat(),'outer_years':[2019,2020,2021,2022],
 'inner_alpha_grid':[1.,10.,100.],'label':'5-session adjusted open return','embargo_sessions':5,
 'main_weights':{'trend':.20,'factor':.25,'text':.05,'ML':.25,'RL':.25},'cost_bps_per_leg':5,
 'gross_cap':.8,'single_cap':.15,'industry_cap':.35,'turnover_cap':.25,
 'holdout_status':'not pristine: 2023–2026 previously displayed; 2021–2022 reused in development','selection':'inner past-year MSE only'}
NESTED_TRIALS10=[];PRED10=[];OUTER_MODELS10={};SPLIT_AUDIT10=[]
def cutoff10(start):
    k=OPEN.index.searchsorted(pd.Timestamp(start));d=OPEN.index[max(0,k-5)]
    return (d+pd.Timedelta(hours=9,minutes=30)).tz_localize(NY)
for year in CAPSTONE_CONFIG['outer_years']:
    outer_start=pd.Timestamp(year,1,1);outer_end=pd.Timestamp(year+1,1,1);inner_start=pd.Timestamp(year-1,1,1)
    inner_cut=cutoff10(inner_start);outer_cut=cutoff10(outer_start)
    inner_train=D5[(D5.date<inner_start)&(D5.label_end5<inner_cut)].copy()
    inner_valid=D5[(D5.date>=inner_start)&(D5.date<outer_start)&(D5.label_end5<outer_cut)].copy()
    outer_train=D5[(D5.date<outer_start)&(D5.label_end5<outer_cut)].copy()
    outer_test=D5[(D5.date>=outer_start)&(D5.date<outer_end)].copy();scores=[]
    for alpha in CAPSTONE_CONFIG['inner_alpha_grid']:
        try:
            model=regression_pipe(FEATURES,Ridge(alpha=alpha));model.fit(inner_train,inner_train.y5);p=model.predict(inner_valid);mse=mean_squared_error(inner_valid.y5,p)
            scores.append((mse,alpha));NESTED_TRIALS10.append({'outer_year':year,'alpha':alpha,'inner_MSE':mse,'status':'completed','inner_train_n':len(inner_train),'inner_valid_n':len(inner_valid)})
        except Exception as e:NESTED_TRIALS10.append({'outer_year':year,'alpha':alpha,'status':'failed','error':str(e)})
    assert len(scores)==3,'Candidate failure must be investigated'
    _,alpha=min(scores,key=lambda z:(z[0],z[1]));model=regression_pipe(FEATURES,Ridge(alpha=alpha));model.fit(outer_train,outer_train.y5);OUTER_MODELS10[year]=model
    p=model.predict(outer_test);frame=outer_test[['date','ticker','decision_at','execution_session','execution_at','label_end5','y5']].copy();frame['prediction']=p;frame['outer_year']=year;frame['chosen_alpha']=alpha;PRED10.append(frame)
    SPLIT_AUDIT10.append({'year':year,'inner_fit_last_label':inner_train.label_end5.max(),'inner_valid_start':inner_valid.decision_at.min(),
      'outer_fit_last_label':outer_train.label_end5.max(),'outer_decision_start':outer_test.decision_at.min(),'embargo_cutoff':outer_cut,
      'alpha':alpha,'outer_n':len(outer_test),'unique_dates':outer_test.date.nunique(),**prediction_stats(outer_test,p)})
PRED10=pd.concat(PRED10,ignore_index=True).sort_values(['date','ticker']);TRIAL_TABLE10=pd.DataFrame(NESTED_TRIALS10);SPLIT_TABLE10=pd.DataFrame(SPLIT_AUDIT10)
display(pd.DataFrame([CAPSTONE_CONFIG]));display(TRIAL_TABLE10);display(SPLIT_TABLE10)
print('10.01 PASS: all alpha candidates retained, mature labels, five-session embargo, whole-date outer folds')


In [ ]:
# 10.02 同资产先合成仓位再记账，主线消融与风险匹配
DATES10=OPEN.loc['2019-01-02':'2022-12-29'].index
SLEEVE10={n:pd.DataFrame(0.,index=DATES10,columns=ASSETS) for n in ['trend','factor','text','ML','RL']}
trend10=(CLOSE[ASSETS]>CLOSE[ASSETS].rolling(200).mean()).astype(float).shift(1).reindex(DATES10).fillna(0)
SLEEVE10['trend']=trend10.div(trend10.sum(axis=1).replace(0,np.nan),axis=0).fillna(0)*.8
SLEEVE10['factor']=L3_TARGETS['equal'].reindex(DATES10).fillna(0)
SLEEVE10['text'].loc[:,'IBM']=L4_TARGETS['price+LM'].IBM.reindex(DATES10).fillna(0)
# 嵌套模型的逐期预测，每5个合格决策日生成下一开盘起五日仓位。
decisions10=pd.DatetimeIndex(sorted(PRED10.date.unique()))[::5]
for d,g in PRED10[PRED10.date.isin(decisions10)].groupby('date'):
    chosen=g[g.prediction>0].sort_values(['prediction','ticker']).tail(3)
    if len(chosen)==0:continue
    entry=g.execution_session.iloc[0];j=OPEN.index.get_loc(entry);window=OPEN.index[j:j+5].intersection(DATES10)
    SLEEVE10['ML'].loc[window,chosen.ticker]=.8/len(chosen)
SLEEVE10['RL']=pd.concat([ENSEMBLE8['validation'][1],ENSEMBLE8['development_eval'][1]]).reindex(DATES10).fillna(0)
BASE10=sum(SLEEVE10[n]*weight for n,weight in CAPSTONE_CONFIG['main_weights'].items())
STATE_MULT10=PREV6.reindex(DATES10).map(SIG6.state_active).fillna(False).astype(float)*.75+.25
RAW10={'full':BASE10.mul(STATE_MULT10,axis=0),'no_state_filter':BASE10.copy()}
for dropped in ['text','ML','RL']:
    RAW10['no_'+dropped]=(BASE10-SLEEVE10[dropped]*CAPSTONE_CONFIG['main_weights'][dropped]).mul(STATE_MULT10,axis=0)
RAW10['equal_reference']=DESIRED6['equal'].reindex(DATES10).fillna(0)
def repair10(target):
    w=target.clip(lower=0).copy();scale=np.minimum(1.,.8/w.sum(axis=1).clip(lower=1e-12))
    scale=np.minimum(scale,.15/w.max(axis=1).clip(lower=1e-12))
    for j in GROUP6.values():scale=np.minimum(scale,.35/w.iloc[:,j].sum(axis=1).clip(lower=1e-12))
    return w.mul(scale,axis=0)
TARGETS10={};LEDGERS10={};TRADES10={};LIMIT10={}
def run10(name,raw,commission=3,slippage=2):
    w,audit=limited_targets(OPEN[ASSETS],repair10(raw),cost=(commission+slippage)/10000)
    x,t=l2_simulate_multi(OPEN[ASSETS],w,commission=commission,slippage=slippage,borrow_rate=0,funding=0,initial=10000)
    TARGETS10[name]=w;LEDGERS10[name]=x;TRADES10[name]=t;LIMIT10[name]=audit
for n,w in RAW10.items():run10(n,w)
RISK_MATCH10=[]
for name in ['no_text','no_ML','no_RL','no_state_filter']:
    reference=LEDGERS10['full'].loc['2019':'2020'].net_return.std();den=LEDGERS10[name].loc['2019':'2020'].net_return.std();scale=min(1.,reference/max(den,1e-12))
    RISK_MATCH10.append({'ablation':name,'validation_frozen_scale':scale})
    run10(name+'_riskmatched',RAW10[name]*scale)
for name,cost in [('full_zero_cost',0),('full_10bp',10)]:run10(name,RAW10['full'],commission=cost,slippage=0)
RESULT10=pd.DataFrame([{'model':n,'period':period,**ledger_stats(x.loc[a:b]),'volatility':x.loc[a:b].net_return.std()*np.sqrt(252),'avg_gross':x.loc[a:b].gross_exposure.mean()}
 for n,x in LEDGERS10.items() for period,a,b in [('validation_reused','2019-01-02','2020-12-31'),('frozen_development_diagnostic','2021-01-01','2022-12-29')]]).set_index(['period','model'])
display(pd.DataFrame(RISK_MATCH10));display(RESULT10)
print('Removed sleeves are not renormalized; exposure reduction shown. Risk matching freezes validation scale and cannot guarantee same future vol.')
fig,ax=plt.subplots(1,2,figsize=(14,4))
for n in ['full','no_text','no_ML','no_RL','no_state_filter','equal_reference']:
    x=LEDGERS10[n].loc['2021':];wealth=(1+x.net_return).cumprod();ax[0].plot(x.index,wealth,label=n);ax[1].plot(x.index,wealth/wealth.cummax().clip(lower=1)-1,label=n)
ax[0].legend(fontsize=7);ax[0].set_title('Frozen development diagnostic: common-account net wealth');ax[1].set_title('Same-capital drawdown');plt.tight_layout();plt.show()
print('10.02 PASS: raw asset targets combined before one fee calculation; intraday/option reports remain separate')


In [ ]:
# 10.03 配对区块bootstrap、DSR敏感性与负结果台账
from scipy.stats import skew,kurtosis
DEV_RET10=pd.DataFrame({n:x.net_return.loc['2021-01-01':] for n,x in LEDGERS10.items()}).dropna()
BOOT10=[];rng10=np.random.default_rng(10001)
for comparator in ['equal_reference','no_text','no_ML','no_RL','no_state_filter']:
    difference=(DEV_RET10['full']-DEV_RET10[comparator]).to_numpy();n=len(difference)
    for block in [5,21,63]:
        starts=rng10.integers(0,n,size=(1000,int(np.ceil(n/block))))
        indices=((starts[:,:,None]+np.arange(block)[None,None,:])%n).reshape(1000,-1)[:,:n]
        samples=difference[indices].mean(axis=1)
        BOOT10.append({'comparison':'full minus '+comparator,'block_sessions':block,'replications':1000,'mean_daily_difference':difference.mean(),
          'bootstrap_q025':np.quantile(samples,.025),'bootstrap_q975':np.quantile(samples,.975),'fraction_bootstrap_le_zero':np.mean(samples<=0),
          'interpretation':'descriptive paired circular block bootstrap, not independent final test'})
BOOT_TABLE10=pd.DataFrame(BOOT10);display(BOOT_TABLE10)
# 已登记的试验下界：保留种子、数据处理、模型、成本与组合实验，不选择性删除失败。
TRIAL_CATALOG10=[]
for lesson,objects in [(1,experiment_registry),(2,L2_EXPERIMENTS)]:
    for i,obj in enumerate(objects):TRIAL_CATALOG10.append({'lesson':lesson,'trial_id':str(i),'detail':str(obj)})
for name in L3_LEDGERS:TRIAL_CATALOG10.append({'lesson':3,'trial_id':name,'detail':'factor score or tercile'})
for name in L4_MODELS:TRIAL_CATALOG10.append({'lesson':4,'trial_id':name,'detail':'event classifier'})
for name in L5_LEDGERS:TRIAL_CATALOG10.append({'lesson':5,'trial_id':name,'detail':'data, model or ablation account'})
for name in L6_LEDGERS:TRIAL_CATALOG10.append({'lesson':6,'trial_id':name,'detail':'risk allocation'})
for row in TRAIN_LOG8:TRIAL_CATALOG10.append({'lesson':8,'trial_id':str(row),'detail':'training seed or reward variant, includes failures'})
for row in NESTED_TRIALS10:TRIAL_CATALOG10.append({'lesson':10,'trial_id':str(row),'detail':'inner candidate'})
for name in LEDGERS10:TRIAL_CATALOG10.append({'lesson':10,'trial_id':name,'detail':'main ablation or cost account'})
TRIAL_CATALOG10=pd.DataFrame(TRIAL_CATALOG10);KNOWN_TRIALS10=len(TRIAL_CATALOG10)
# PSR/DSR使用日Sharpe和非超额峰度；基于iid公式，序列相关限制必须保留。
def dsr10(returns,trial_count,dispersion):
    x=np.asarray(returns,float);n=len(x);s=x.mean()/max(x.std(ddof=1),1e-12);g=skew(x,bias=False);k=kurtosis(x,fisher=False,bias=False)
    euler=.5772156649015329
    benchmark=dispersion*((1-euler)*norm.ppf(1-1/trial_count)+euler*norm.ppf(1-1/(trial_count*np.e))) if trial_count>1 else 0.
    denominator=np.sqrt(max(1-g*s+(k-1)*s*s/4,1e-12))
    return {'daily_SR':s,'annual_SR_252':s*np.sqrt(252),'daily_SR_benchmark':benchmark,'skewness':g,'nonexcess_kurtosis':k,
      'iid_DSR_diagnostic':norm.cdf((s-benchmark)*np.sqrt(n-1)/denominator),'n_days':n,'trial_count_assumed_independent':trial_count}
sharpes=DEV_RET10.mean()/DEV_RET10.std(ddof=1);dispersion=float(sharpes.std(ddof=1));DSR_ROWS10=[]
for trials in [KNOWN_TRIALS10,2*KNOWN_TRIALS10,5*KNOWN_TRIALS10]:
    DSR_ROWS10.append({'model':'full','known_trial_lower_bound':KNOWN_TRIALS10,'SR_dispersion_among_shown_accounts':dispersion,**dsr10(DEV_RET10.full,trials,dispersion)})
DSR_TABLE10=pd.DataFrame(DSR_ROWS10);display(DSR_TABLE10)
print('DSR assumes independent trial counts and iid-return asymptotics; shown strategies correlate, unknown older trials and serial dependence prevent a definitive significance claim.')
NEGATIVE_RESULTS10=[]
full=RESULT10.loc[('frozen_development_diagnostic','full')]
for name in ['no_text','no_ML','no_RL','no_state_filter','equal_reference']:
    row=RESULT10.loc[('frozen_development_diagnostic',name)]
    NEGATIVE_RESULTS10.append({'hypothesis':'full improves net return vs '+name,'delta_net_return':full.net_return-row.net_return,
      'observed_result':'not_supported_in_this_window' if full.net_return<=row.net_return else 'positive_descriptive_difference_only',
      'reject_reason_or_limit':'development data already observed; no independent final holdout; exposure and costs differ'})
NEGATIVE_TABLE10=pd.DataFrame(NEGATIVE_RESULTS10);display(NEGATIVE_TABLE10)
print('10.03 PASS: paired dependence-aware diagnostic, explicit multiple-trial sensitivity, unsuccessful hypotheses retained')


In [ ]:
# 10.04 全链路防泄漏、费用、约束与数值验收
L10_CHECKS=[]
def check10(name,ok):
    assert bool(ok),name;L10_CHECKS.append({'check':name,'status':'PASS'})
for lesson in [2,3,4,5,6,7,8,9]:
    checks=globals()['L'+str(lesson)+'_VALIDATION'];check10('lesson '+str(lesson)+' completed checks',len(checks)>0 and checks.status.eq('PASS').all())
check10('outer day groups unique',PRED10.groupby('date').outer_year.nunique().max()==1)
check10('outer asset-date unique',not PRED10.duplicated(['date','ticker']).any())
check10('all outer labels mature before embargo',(SPLIT_TABLE10.outer_fit_last_label<SPLIT_TABLE10.embargo_cutoff).all())
check10('inner training before validation',(SPLIT_TABLE10.inner_fit_last_label<SPLIT_TABLE10.inner_valid_start).all())
check10('full outer predictions strictly prior to execution',(PRED10.decision_at<PRED10.execution_at).all())
check10('all inner trials kept',len(TRIAL_TABLE10)==12 and TRIAL_TABLE10.status.eq('completed').all())
check10('trial lower bound disclosed',KNOWN_TRIALS10>=len(LEDGERS10)+len(TRAIN_LOG8)+12)
check10('bootstrap interval ordered',(BOOT_TABLE10.bootstrap_q025<=BOOT_TABLE10.bootstrap_q975).all())
check10('DSR probability range',DSR_TABLE10.iid_DSR_diagnostic.between(0,1).all())
check10('DSR more assumed trials never helps',DSR_TABLE10.iid_DSR_diagnostic.is_monotonic_decreasing)
check10('no pristine holdout claim','not pristine' in CAPSTONE_CONFIG['holdout_status'])
check10('no real HFT claim',LOB_MANIFEST['source_kind']=='SYNTHETIC_MECHANISM_ONLY' or LOB_MANIFEST['source_kind']=='official_single_day_sample')
check10('actual TPU arrays and checkpoints',mc_price9.devices()=={TPU} and len(JAX_MODELS8)==10 and len(CHECKPOINTS8)==15)
for name,x in LEDGERS10.items():
    w=TARGETS10[name];audit=LIMIT10[name]
    check10(name+' one shared window',x.index.equals(DATES10))
    check10(name+' equity accounting',np.allclose(x.net_pnl,x.market_pnl-x.transaction_cost-x.borrow_cost-x.funding_cost,atol=1e-6))
    check10(name+' compounded return',np.allclose(x.wealth,(1+x.net_return).cumprod()))
    check10(name+' weight constraints',w.min().min()>=-1e-8 and w.sum(axis=1).max()<=.800001 and w.max().max()<=.150001 and all(w.iloc[:,j].sum(axis=1).max()<=.350001 for j in GROUP6.values()))
    check10(name+' nonnegative costs',(x[['transaction_cost','borrow_cost','funding_cost']]>=0).all().all())
    check10(name+' turnover and forced-risk audit',((x.turnover<=.250001)|audit.startup|audit.forced_risk_override).all())
check10('zero-cost account pays no fees',np.allclose(LEDGERS10['full_zero_cost'].transaction_cost,0))
# 冻结模型只做transform不能悄悄fit；极端未来输入不改变训练填补统计。
model=OUTER_MODELS10[2022];imputer=model.named_steps['features'].named_transformers_['numeric'].named_steps['impute'];before=imputer.statistics_.copy()
probe=D5[D5.date.dt.year.eq(2022)].head(24).copy();probe[FEATURES]=probe[FEATURES]*10+100;model.predict(probe)
check10('outer transform never refits',np.array_equal(before,imputer.statistics_,equal_nan=True))
# 未来价格扰动不改变决策时的prefix return；来源审计和版本保留。
d=PRED10.date.iloc[0];a=PRED10.ticker.iloc[0];past=CLOSE.loc[:d,a];prefix=past.iloc[-1]/past.iloc[-6]-1
perturbed=CLOSE[a].copy();perturbed.loc[perturbed.index>d]*=10
check10('future perturbation does not alter historical return',np.isclose(prefix,perturbed.loc[:d].iloc[-1]/perturbed.loc[:d].iloc[-6]-1))
L10_VALIDATION=pd.DataFrame(L10_CHECKS);display(L10_VALIDATION)
ALL_VALIDATION10=pd.concat([globals()['L'+str(i)+'_VALIDATION'].assign(lesson=i) for i in range(2,11)],ignore_index=True)
print('Verified total recorded checks (lessons2–10)',len(ALL_VALIDATION10));display(ALL_VALIDATION10.groupby('lesson').size().rename('passed_checks'))
L10_CARD={'lesson':10,'status':'verified','checks':len(L10_CHECKS),'outer_years':CAPSTONE_CONFIG['outer_years'],
 'main_account_count':len(LEDGERS10),'known_trial_lower_bound':KNOWN_TRIALS10,'historical_holdout':CAPSTONE_CONFIG['holdout_status'],'completed_utc':pd.Timestamp.now(tz='UTC').isoformat()}
print('L10 RESEARCH FINAL PASS',json.dumps(L10_CARD,ensure_ascii=False))
